# Examen de diseño de red 5G

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ollerenac/wireless-communication-systems/blob/main/docs/sessions/07-network-design/examen-alumno.ipynb)

**Alumno:** _________________ · **Mapa:** elige `ESCENA` en la primera celda · **Duración:** 3 h

**Material permitido:** los apuntes de la lección
([Sesión 07](https://ollerenac.github.io/wireless-communication-systems/sessions/07-network-design/))
y este notebook. Las **pistas** de cada fase te dicen qué tabla o sección
consultar.

**Cómo se califica:** cada número que escribas debe tener **origen**: una
oferta de la publicidad, una cláusula del contrato, o una tabla de la
lección (citada). Un valor distinto al de la pauta pero bien defendido; un valor "correcto" sin defender, no.

**Dinámica de cada fase** — 5 bloques:

1. **Enunciado**: la parte del proyecto que esta fase resuelve.
2. **Pista**: qué tabla/sección de la lección consultar.
3. **Celda `TU TRABAJO`**: complétala (los `None` son tuyos).
4. **Celda `VERIFICADOR`**: ejecútala sin modificarla — chequea que tu
   respuesta esté bien *formada* (no que esté bien *pensada*).
5. **Celda `JUSTIFICACIÓN`**: responde en 2–3 líneas por pregunta.

---


In [ ]:
# ---- Preparación del entorno (ejecutar SIEMPRE esta celda primero) ----
# ESCENA: elige exactamente uno de los dos mapas disponibles.
ESCENA = "jesus-maria-01" # @param ["jesus-maria-01", "san-isidro-01"]

ESCENARIOS = {
    "jesus-maria-01": {"bw_hz": 80e6,  "max_sitios": 7, "p_tx_dbm_max": 43.0, "n_prb": 217},
    "san-isidro-01":   {"bw_hz": 100e6, "max_sitios": 6, "p_tx_dbm_max": 43.0, "n_prb": 273},
}
if ESCENA not in ESCENARIOS:
    raise ValueError(f"ESCENA debe ser una de {list(ESCENARIOS)}; recibí {ESCENA!r}")
ESCENARIO = ESCENARIOS[ESCENA]

import importlib.util, os, subprocess, sys
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ModuleNotFoundError:
    EN_COLAB = False

if EN_COLAB and importlib.util.find_spec("sionna") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sionna-rt"])

ruta_xml = Path("escenas") / ESCENA / f"{ESCENA}.xml"
if not ruta_xml.exists():
    url = ("https://ollerenac.github.io/wireless-communication-systems/"
           f"sessions/07-network-design/escenas/{ESCENA}.zip")
    archivo_zip = Path(f"{ESCENA}.zip")
    urlretrieve(url, archivo_zip)
    with ZipFile(archivo_zip) as zf:
        esperado = f"{ESCENA}/{ESCENA}.xml"
        assert esperado in zf.namelist(), f"el ZIP no contiene {esperado}"
        zf.extractall("escenas")
    archivo_zip.unlink()

os.makedirs("resultados", exist_ok=True)
assert ruta_xml.exists(), f"no se encontró {ruta_xml}"
entorno = "Colab" if EN_COLAB else "Entorno local"
print(f"{entorno} listo: mapa '{ESCENA}' disponible en {ruta_xml}.")



## El proyecto

Un operador móvil entrante te contrata para diseñar su red de acceso 5G
sobre uno de los siguientes mapas 3D de Lima. Trabaja únicamente con
el escenario que elegiste en `ESCENA`.

### `jesus-maria-01` — Jesús María

Polígono de **1.27 × 1.07 km** de distrito residencial denso, con el eje
hospitalario de la Av. Arenales dentro del área. Su publicidad dice:

> *"5G real en todo Jesús María. Video HD sin cortes y videollamadas que
> no se caen, en tu casa y en la calle. La red que sí llega."*

Términos de referencia (TdR):

- Licencia: **80 MHz en n78** (3.5 GHz), TDD.
- Azoteas disponibles; el municipio autoriza **máximo 7 sitios**.
- Potencia máxima por sector: **43 dBm**.
- Mercado: ~**30 000 hab/km²**, participación objetivo **25%**,
  consumo típico **12 GB/mes** por abonado.
- Target de borde según la guía de planificación del operador:
  **50 Mbps DL / 5 Mbps UL** en el percentil 5 (ref. NGMN
  *"50 Mbps everywhere"*).

### `san-isidro-01` — San Isidro

Polígono de **1.2 × 1.1 km** del distrito financiero. Su publicidad dice:

> *"Videollamadas nítidas y tu oficina en la nube, en todo San Isidro."*

Términos de referencia (TdR):

- Licencia: **100 MHz en n78** (3.5 GHz), TDD.
- Azoteas disponibles; el municipio autoriza **máximo 6 sitios**.
- Potencia máxima por sector: **43 dBm**.
- Mercado: ~**25 000 personas presentes/km²** en hora cargada,
  participación objetivo **30%**, consumo típico **10 GB/mes** por abonado.
- Target de borde según la guía de planificación del operador:
  **50 Mbps DL / 5 Mbps UL** en el percentil 5 (ref. NGMN
  *"50 Mbps everywhere"*).


---

## Fase 0 — Requisitos del proyecto

**Enunciado.** Traduce la publicidad y los términos de referencia a un
diccionario `REQ` medible. Toda oferta debe convertirse en **número con
probabilidad**; todo número que declares, el trazador lo va a cobrar en
la Fase 6.

**Pistas.**

- La oferta de cobertura *"en todo"* tu distrito → ¿qué umbral RSRP y qué probabilidad? →
  **Tabla 0.1** y **§0.1** (umbral y probabilidad de control).
- *¿cuánto SINR necesitan los datos y por qué ese umbral?* → **Tabla 0.2**
  y **§0.2** (umbral y probabilidad de datos).
- El **target de borde te lo dan los TdR** (así llega en la práctica:
  de la guía de planificación del operador). Tu trabajo no es elegirlo:
  es registrarlo y **verificar que cubre el mínimo técnico** — el mayor DL y el
  mayor UL de la **Tabla 0.3** sobre los servicios que TU publicidad
  ofrece (§0.3).
- Capacidad agregada → la cadena GB/mes → kbps de hora cargada está en
  **§0.4** (Tabla 0.4 la tabula para volúmenes típicos).
- La estructura de un `REQ` completo (qué requisitos existen) → **Tabla
  0.5** (§0.5).


In [ ]:
# ================= TU TRABAJO — completa cada None =================
# Cada número debe tener origen (publicidad, TdR, o tabla citada).
import math

# --- Primero, la demanda de hora cargada (fórmula en §0.4) ---
personas_km2     = None   # <- COMPLETA: de los TdR
market_share     = None   # <- COMPLETA: de los TdR (fracción)
gb_mes           = None   # <- COMPLETA: de los TdR
f_bh             = None   # <- COMPLETA: fracción de tráfico en hora cargada (§0.4: rango típico)

kbps_por_abonado = None   # <- COMPLETA: escribe aquí tu cálculo con la fórmula de §0.4
demanda_mbps_km2 = None   # <- COMPLETA: abonados/km2 x kbps_por_abonado, en Mbps/km2

# --- Luego, el contrato del diseño: R5 sale de tu demanda, no al revés ---
REQ = {
    # R1 — área de servicio: el mapa asignado (la escena no la toques)
    "escena":            f"escenas/{ESCENA}/{ESCENA}.xml",
    "area_km2":          None,   # <- COMPLETA: dimensiones de TU mapa
    # R2 — cobertura de control: la oferta de cobertura de TU publicidad
    "rsrp_min_dbm":      None,   # <- COMPLETA: sección 0.1
    "rsrp_prob":         None,   # <- COMPLETA: sección 0.1 (fracción, 0 a 1)
    # R3 — calidad de datos: los servicios de TU publicidad
    "sinr_min_db":       None,   # <- COMPLETA: sección 0.2
    "sinr_prob":         None,   # <- COMPLETA: sección 0.2
    # R4 — throughput de borde (percentil 5 del área)
    "thr_borde_dl_mbps": None,   # <- COMPLETA: de los TdR — verifica el mínimo técnico (Tabla 0.3, §0.3)
    "thr_borde_ul_mbps": None,   # <- COMPLETA: de los TdR — verifica el mínimo técnico (Tabla 0.3, §0.3)
    # R5 — capacidad agregada: tu demanda calculada + el margen que declares
    "capacidad_mbps_km2": None,   # <- COMPLETA: usa demanda_mbps_km2 (sección 0.4)
    # (R6 — servicios y latencia: se garantiza por arquitectura/QoS,
    #  no hay mapa del trazador que lo verifique -> no entra al dict)
    # R7 — espectro licenciado
    "banda":             None,   # <- COMPLETA: texto, p. ej. "n78"
    "fc_hz":             None,   # <- COMPLETA
    "bw_hz":             None,   # <- COMPLETA
    # R8 — restricciones de despliegue
    "max_sitios":        None,   # <- COMPLETA
    "p_tx_dbm_max":      None,   # <- COMPLETA
}

# (guardia: solo detecta None pendientes — estos None no se completan)
if None in (kbps_por_abonado, demanda_mbps_km2, REQ["capacidad_mbps_km2"]):
    print("(hay None pendientes — completa y vuelve a ejecutar)")
else:
    print(f"{kbps_por_abonado:.0f} kbps/abonado -> demanda {demanda_mbps_km2:.0f} Mbps/km2 "
          f"-> tu R5: {REQ['capacidad_mbps_km2']:.0f}")

In [ ]:
# ============ VERIFICADOR — ejecuta esta celda SIN modificarla ============
# Chequea que tu REQ esté bien FORMADO. No chequea que esté bien PENSADO:
# un valor absurdo puede pasar aquí y reprobar en la Fase 6.

CLAVES = {"escena", "area_km2", "rsrp_min_dbm", "rsrp_prob", "sinr_min_db",
          "sinr_prob", "thr_borde_dl_mbps", "thr_borde_ul_mbps",
          "capacidad_mbps_km2", "banda", "fc_hz", "bw_hz",
          "max_sitios", "p_tx_dbm_max"}

faltan = CLAVES - set(REQ)
assert not faltan, f"faltan claves en REQ: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES if REQ.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

# rangos de cordura (anchos a propósito — no son la respuesta)
assert -140 < REQ["rsrp_min_dbm"] < -60,  "RSRP fuera de rango físico razonable"
assert 0 < REQ["rsrp_prob"] <= 1,          "rsrp_prob es fracción (0 a 1)"
assert 0 < REQ["sinr_prob"] <= 1,          "sinr_prob es fracción (0 a 1)"
assert REQ["thr_borde_ul_mbps"] < REQ["thr_borde_dl_mbps"], "¿UL mayor que DL?"

# términos de referencia (esto sí es el enunciado)
assert REQ["bw_hz"] <= ESCENARIO["bw_hz"], \
    f"la licencia de {ESCENA} es de {ESCENARIO['bw_hz']/1e6:.0f} MHz — no puedes usar más"
assert REQ["thr_borde_dl_mbps"] == 50.0 and REQ["thr_borde_ul_mbps"] == 5.0, \
    "el target de borde lo fija la guía del operador (está en los TdR)"
assert REQ["max_sitios"] <= ESCENARIO["max_sitios"], \
    f"el municipio autoriza máximo {ESCENARIO['max_sitios']} sitios en {ESCENA}"
assert REQ["p_tx_dbm_max"] <= ESCENARIO["p_tx_dbm_max"], \
    f"la potencia máxima por sector es {ESCENARIO['p_tx_dbm_max']:.0f} dBm"

# coherencia de R5 con tu propio cálculo
assert None not in (kbps_por_abonado, demanda_mbps_km2), "completa el cálculo de R5"
assert 0.05 <= f_bh <= 0.20, \
    "f_bh fuera del rango razonable de planificación — revisa §0.4"
assert REQ["capacidad_mbps_km2"] >= demanda_mbps_km2, \
    "R5 no cubre la demanda que tú mismo calculaste — el contrato nace roto"

print("REQ bien formado ✓ — la defensa de los valores es tuya (justificación)")


**JUSTIFICACIÓN — responde aquí mismo (edita esta celda):**

1. ¿De qué fila de la Tabla 0.1 sale tu umbral RSRP, y por qué esa fila
   traduce la oferta de cobertura *"en todo"* tu distrito?

   _tu respuesta..._

2. Según la Tabla 0.3, ¿qué bitrate mínimo exigen en DL y en UL los
   servicios que ofrece TU publicidad (el más exigente de cada columna)?
   ¿El target de los TdR (50/5) los cubre? ¿Por qué crees que 50 es
   mucho mayor que ese mínimo?

   _tu respuesta..._

3. ¿Qué margen dejaste entre la demanda calculada y tu R5? ¿Qué pasaría
   con tu diseño si la participación de mercado sube diez puntos porcentuales?

   _tu respuesta..._

4. La publicidad ofrece videollamadas "que no se caen" y la Tabla 0.3
   exige < 150 ms de latencia para ese servicio. ¿Por qué ninguna celda
   de este notebook puede verificar esa promesa, y qué parte de la red
   la garantiza?

   _tu respuesta..._


---

## Fase 1 — Análisis del espectro licenciado

**Enunciado.** Los TdR fijan tu banda y ancho de canal (R7, n78 TDD). Aquí no
eliges — **cuantificas** lo que esa licencia implica, porque las Fases 2
y 3 heredan estos números.

**Pistas.**

- La física de tu banda → **§1.1** (el cálculo de Δpérdida → radio →
  sitios; replícalo con TU frecuencia de referencia).
- Los PRB de TU canal → **§1.4**. La guarda para tu ancho y SCS está en
  TS 38.101-1, Tabla 5.3.3-1: **0.925 MHz por borde** para 80 MHz y
  **0.845 MHz por borde** para 100 MHz, ambos con SCS de 30 kHz. La
  división ingenua del ancho total da un número equivocado.
- El reparto TDD → **§1.3** (patrón DDDSU y sus fracciones de tiempo).


In [ ]:
# ============ TU TRABAJO (Fase 1) — completa cada None ============
import math

# --- 1. La física de tu banda (§1.1) ---
N_PROP = 3.8                       # exponente de propagación urbano denso
bandas_hz = {"n28 (700 MHz)": 0.7e9, "n1 (2.1 GHz)": 2.1e9,
             "n78 (3.5 GHz)": 3.5e9, "n258 (26 GHz)": 26e9}
fc_ref = REQ["fc_hz"]
print(f"{'banda':<15} {'Δpérdida':>9} {'radio rel.':>10} {'sitios rel.':>11}")
for nombre, f in bandas_hz.items():
    delta_db   = None   # <- COMPLETA: Δ vs tu banda — fórmula en §1.1
    r_rel      = None   # <- COMPLETA: radio relativo con MAPL fijo — fórmula en §1.1
    sitios_rel = None   # <- COMPLETA: sitios para la misma área — fórmula en §1.1
    # (este if solo evita que el print se ejecute con None pendientes — no toques el siguiente None)
    if None in (delta_db, r_rel, sitios_rel):
        print(f"{nombre:<15}  (completa las fórmulas)")
    else:
        print(f"{nombre:<15} {delta_db:>+7.1f}dB {r_rel:>9.2f}x {sitios_rel:>10.2f}x")

# --- 2. PRBs de TU canal (§1.4) ---
scs_khz    = None   # <- COMPLETA: ¿la eliges tú? — Tabla 1.3 y §1.4
guarda_mhz = None   # <- COMPLETA: guarda por borde del canal — método en §1.4; el valor para tu ancho está en la pista
bw_mhz     = REQ["bw_hz"] / 1e6
bw_tx_mhz  = None   # <- COMPLETA: ancho transmisible = canal − 2 × guarda (§1.4)
n_prb      = None   # <- COMPLETA: cuántos PRB caben en bw_tx; 1 PRB = 12 subportadoras × SCS = 0.36 MHz (§1.4); entero hacia abajo

# --- 3. Reparto TDD DDDSU (§1.3) ---
f_dl = None   # <- COMPLETA: cuenta los símbolos del patrón DDDSU — aritmética en §1.3
f_ul = None   # <- COMPLETA: ídem — §1.3

ESPECTRO = {
    "scs_khz":     scs_khz,
    "n_prb":       n_prb,
    "f_dl":        f_dl,
    "f_ul":        f_ul,
    "b_dl_ef_mhz": None,   # <- COMPLETA: MHz "efectivos" de bajada
    "b_ul_ef_mhz": None,   # <- COMPLETA
}
# (guardia: solo detecta None pendientes — estos None no se completan)
if None in ESPECTRO.values():
    print("\n(hay None pendientes — completa y vuelve a ejecutar)")
else:
    print(f"\ncanal {bw_mhz:.0f} MHz -> {n_prb} PRB | "
          f"efectivos DL {ESPECTRO['b_dl_ef_mhz']:.1f} / UL {ESPECTRO['b_ul_ef_mhz']:.1f} MHz")

In [ ]:
# ======== VERIFICADOR (Fase 1) — ejecuta esta celda SIN modificarla ========
# Chequea que tu ESPECTRO esté bien FORMADO y coherente con el estándar.

CLAVES_E = {"scs_khz", "n_prb", "f_dl", "f_ul", "b_dl_ef_mhz", "b_ul_ef_mhz"}
faltan = CLAVES_E - set(ESPECTRO)
assert not faltan, f"faltan claves en ESPECTRO: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES_E if ESPECTRO.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

# el estándar (esto sí es dato, como las cláusulas de los TdR)
assert ESPECTRO["scs_khz"] == 30, \
    "con n78 la numerología del proyecto queda fijada en los TdR — revisa §1.4"
assert ESPECTRO["n_prb"] == ESCENARIO["n_prb"], \
    "no coincide con TS 38.101-1 Tabla 5.3.2-1 — revisa las guardas (§1.4)"

# reparto TDD razonable para DDDSU
assert 0.60 <= ESPECTRO["f_dl"] <= 0.80, "fracción DL fuera de DDDSU (§1.3)"
assert 0.15 <= ESPECTRO["f_ul"] <= 0.30, "fracción UL fuera de DDDSU (§1.3)"

# coherencia de los anchos efectivos con TU licencia
bw_mhz = REQ["bw_hz"] / 1e6
assert abs(ESPECTRO["b_dl_ef_mhz"] - bw_mhz * ESPECTRO["f_dl"]) < 1.0, \
    "b_dl_ef no es ancho x fracción DL"
assert abs(ESPECTRO["b_ul_ef_mhz"] - bw_mhz * ESPECTRO["f_ul"]) < 1.0, \
    "b_ul_ef no es ancho x fracción UL"

print("ESPECTRO bien formado ✓ — la Fase 2 hereda estos números")


**JUSTIFICACIÓN (Fase 1) — responde aquí mismo:**

1. Con la tabla que calculaste en el bloque 1 de TU TRABAJO (la réplica
   de la Tabla 1.1 de la lección): ¿cuántos sitios costaría este mismo
   proyecto en n258? ¿Y qué ganarías y qué perderías si la licencia fuera en n28?
   (pista: ¿cuánto espectro contiguo existe en 700 MHz?)

   _tu respuesta..._

2. ¿Por qué tu `n_prb` no sale de dividir directamente el ancho total
   del canal entre 0.36 MHz?

   _tu respuesta..._

3. ¿Cuántos MHz efectivos de subida te deja DDDSU, y alcanzan para el
   target de borde UL de tus TdR? ¿Es el tiempo el problema del UL?

   _tu respuesta..._

4. ¿Por qué el SCS no lo elegiste tú?

   _tu respuesta..._


---

## Fase 2 — Dimensionamiento por cobertura

**Enunciado.** Convierte tu presupuesto de enlace en un número de sitios.
Heredas `REQ` (Fase 0) y `ESPECTRO` (Fase 1). Hipótesis declaradas del
curso, ya puestas en el código: $G_{tx} = 16$ dBi, $M_{shadow} = 9$ dB
(σ = 8, 95% de área — Tabla 2.3), UE de 23 dBm / 0 dBi, NF de la BS 5 dB.

**Pistas.**

- EPRE → **§2.1** — con TU potencia y TUS subportadoras, no las de la
  lección.
- Escalera de bajada → **§2.2** (Tabla 2.1; el despeje del MAPL está ahí).
- MAPL → radio → **§2.3** (modelo UMa NLOS; la inversión es un despeje).
- Escalera de subida → **§2.4** — el ancho del UE de borde sale de TU
  requisito R4-UL; la sensibilidad de la BS es el término nuevo.
- ¿Qué enlace define el radio? → **§2.4** (el que soporte menos pérdida).


In [ ]:
# ============ TU TRABAJO (Fase 2) — completa cada None ============
import math

# --- 1. EPRE (§2.1) ---
n_re     = None   # <- COMPLETA: subportadoras de TU canal (usa ESPECTRO)
epre_dbm = None   # <- COMPLETA: tu potencia (REQ) repartida entre n_re — en dB

# --- 2. Escalera de bajada (§2.2): control (SSB/RSRP, cobra R2) ---
g_tx_dbi    = 16.0    # hipótesis declarada del curso (Tabla 2.1)
m_shadow_db = 9.0     # sigma=8 dB, 95% de área (Tabla 2.3)
mapl_dl_db  = None   # <- COMPLETA: despeja la escalera de §2.2 con TU umbral R2

# --- 3. De MAPL a radio (§2.3): UMa NLOS, TR 38.901 ---
fc_ghz = REQ["fc_hz"] / 1e9
d_dl_m = None   # <- COMPLETA: invierte el modelo UMa con tu MAPL de bajada

# --- 4. Escalera de subida (§2.4): datos (cobra R4-UL) ---
bw_ul_hz    = None   # <- COMPLETA: el ancho del UE de borde sale de TU R4-UL (§2.4)
p_ue_dbm    = 23.0    # potencia máxima del UE (clase estándar)
nf_db       = 5.0     # figura de ruido de la BS (hipótesis declarada)
snr_min_db  = 0.0
s_bs_dbm    = None   # <- COMPLETA: sensibilidad de la BS (§2.4: térmico + ventana + NF + SNR)
mapl_ul_db  = None   # <- COMPLETA: escalera de subida (§2.4)
d_ul_m      = None   # <- COMPLETA: invierte el UMa con tu MAPL de subida

# --- 5. Radio de diseño y sitios por cobertura (§2.3, §2.4) ---
radio_m        = None   # <- COMPLETA: ¿cuál de los dos enlaces manda?
area_celda_km2 = None   # <- COMPLETA: hexágono trisectorial: 2.6 x radio² (en km)
n_cobertura    = None   # <- COMPLETA: sitios enteros para tapar TU área (redondea hacia arriba)

COBERTURA = {
    "epre_dbm":    epre_dbm,
    "mapl_dl_db":  mapl_dl_db,
    "mapl_ul_db":  mapl_ul_db,
    "radio_m":     radio_m,
    "n_cobertura": n_cobertura,
}
# (guardia: solo detecta None pendientes — estos None no se completan)
if None in COBERTURA.values():
    print("(hay None pendientes — completa y vuelve a ejecutar)")
else:
    print(f"EPRE {epre_dbm:.1f} dBm | MAPL DL {mapl_dl_db:.1f} / UL {mapl_ul_db:.1f} dB | "
          f"radio {radio_m:.0f} m -> {n_cobertura} sitios por cobertura")

In [ ]:
# ======== VERIFICADOR (Fase 2) — ejecuta esta celda SIN modificarla ========
import math as _m

CLAVES_C = {"epre_dbm", "mapl_dl_db", "mapl_ul_db", "radio_m", "n_cobertura"}
faltan = CLAVES_C - set(COBERTURA)
assert not faltan, f"faltan claves en COBERTURA: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES_C if COBERTURA.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

# rangos físicos de cordura (anchos a propósito)
assert 5 < COBERTURA["epre_dbm"] < 12, \
    "EPRE fuera de rango — ¿repartiste la potencia entre TODAS las subportadoras? (§2.1)"
assert 118 < COBERTURA["mapl_dl_db"] < 132, "MAPL DL fuera de rango — revisa la escalera (§2.2)"
assert 126 < COBERTURA["mapl_ul_db"] < 138, "MAPL UL fuera de rango — revisa la sensibilidad (§2.4)"

# consistencia interna: el radio debe invertir el modelo UMa del enlace limitante
_pl = 13.54 + 39.08*_m.log10(COBERTURA["radio_m"]) + 20*_m.log10(REQ["fc_hz"]/1e9)
assert abs(_pl - min(COBERTURA["mapl_dl_db"], COBERTURA["mapl_ul_db"])) < 0.5, \
    "radio_m no invierte el UMa con el MAPL del enlace limitante (§2.3)"

# consistencia: sitios = área / hexágono trisectorial
_n = _m.ceil(REQ["area_km2"] / (2.6 * (COBERTURA["radio_m"]/1000)**2))
assert COBERTURA["n_cobertura"] == _n, "n_cobertura no sale de tu propio radio (§2.3)"

# restricción del proyecto (R8)
assert COBERTURA["n_cobertura"] <= REQ["max_sitios"], \
    "la cobertura pide más sitios que los que autoriza el municipio — algo anda mal"

print(f"COBERTURA bien formada ✓ — {COBERTURA['n_cobertura']} sitios por cobertura; "
      f"la Fase 3 dirá si la capacidad pide más")


**JUSTIFICACIÓN (Fase 2) — responde aquí mismo:**

1. Tu EPRE dio casi el mismo valor que el de la lección (8.8 dBm), con
   1 dB menos de potencia de amplificador. Muestra la aritmética.

   _tu respuesta..._

2. ¿Qué enlace limita tu celda? ¿Por qué aquí no aplica el dicho "el
   uplink siempre limita", y en qué tipo de proyecto sí aplicaría?

   _tu respuesta..._

3. ¿Qué compra el margen de 9 dB? Si el proyecto exigiera 99% del área
   (Tabla 2.3), ¿cuántos sitios pediría la cobertura y en qué situación
   quedaría el proyecto frente al límite del municipio?

   _tu respuesta..._

4. ¿Para qué sirve el modelo UMa si en la Fase 6 igual vas a trazar
   rayos sobre el mapa real?

   _tu respuesta..._


---

## Fase 3 — Dimensionamiento por capacidad

**Enunciado.** La pregunta gemela de la Fase 2: ¿cuántos sitios para que
la red *aguante* tu demanda de R5? El veredicto del dimensionamiento
combina ambas fases. Hipótesis declaradas, ya puestas en el código:
$SE_{DL} = 2.0$ y $SE_{UL} = 1.2$ bit/s/Hz, $OH = 22\%$, demanda de
subida = 15% de la de bajada.

**Pistas.**

- La planilla de la capacidad → **§3.2**: SE × B × (1−OH) — con TU ancho
  **efectivo** de bajada (Fase 1), no el del canal.
- La demanda → **§3.3**: tu R5 por tu área (Fase 0).
- El veredicto → **§3.3**: los dos dimensionamientos, no uno.
- La subida → **§3.4**: mismo cálculo con los insumos UL; verifica que
  el UL no mande.


In [ ]:
# ============ TU TRABAJO (Fase 3) — completa cada None ============
import math

# --- 1. Capacidad de una celda en bajada (§3.2) ---
se_dl  = 2.0     # bit/s/Hz, promedio de celda SISO (hipótesis declarada)
oh     = 0.22    # overhead de señalización NR
r_celda_dl_mbps = None   # <- COMPLETA: planilla §3.2: se_dl * ESPECTRO["b_dl_ef_mhz"] * (1 - oh)
r_sitio_dl_mbps = None   # <- COMPLETA: un sitio = 3 sectores -> 3 * r_celda_dl_mbps

# --- 2. La demanda contra la que se vende (§3.3) ---
demanda_total_mbps = None   # <- COMPLETA: REQ["capacidad_mbps_km2"] * REQ["area_km2"] (§3.3)
n_capacidad        = None   # <- COMPLETA: entero hacia arriba: math.ceil(demanda / sitio) (§3.3)

# --- 3. El veredicto del dimensionamiento (§3.3) ---
n_sitios = None   # <- COMPLETA: el peor de los dos: max(COBERTURA["n_cobertura"], n_capacidad)

# --- 4. Verificación de la subida (§3.4) ---
se_ul  = 1.2     # promedio de celda en subida (hipótesis declarada)
r_celda_ul_mbps   = None   # <- COMPLETA: misma planilla: se_ul * ESPECTRO["b_ul_ef_mhz"] * (1 - oh)
r_sitio_ul_mbps   = None   # <- COMPLETA: 3 sectores
ul_instalado_mbps = None   # <- COMPLETA: n_sitios * r_sitio_ul_mbps
ul_demanda_mbps   = None   # <- COMPLETA: 0.15 * demanda_total_mbps (hipótesis declarada, §3.4)

CAPACIDAD = {
    "r_celda_dl_mbps":    r_celda_dl_mbps,
    "r_sitio_dl_mbps":    r_sitio_dl_mbps,
    "demanda_total_mbps": demanda_total_mbps,
    "n_capacidad":        n_capacidad,
    "n_sitios":           n_sitios,
    "r_sitio_ul_mbps":    r_sitio_ul_mbps,
    "ul_instalado_mbps":  ul_instalado_mbps,
    "ul_demanda_mbps":    ul_demanda_mbps,
}
# (guardia: solo detecta None pendientes — estos None no se completan)
if None in CAPACIDAD.values():
    print("(hay None pendientes — completa y vuelve a ejecutar)")
else:
    print(f"celda DL {r_celda_dl_mbps:.0f} Mbps | demanda {demanda_total_mbps:.0f} Mbps | "
          f"VEREDICTO: {n_sitios} sitios de {REQ['max_sitios']} autorizados")

In [ ]:
# ======== VERIFICADOR (Fase 3) — ejecuta esta celda SIN modificarla ========
import math as _m

CLAVES_K = {"r_celda_dl_mbps", "r_sitio_dl_mbps", "demanda_total_mbps",
            "n_capacidad", "n_sitios", "r_sitio_ul_mbps",
            "ul_instalado_mbps", "ul_demanda_mbps"}
faltan = CLAVES_K - set(CAPACIDAD)
assert not faltan, f"faltan claves en CAPACIDAD: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES_K if CAPACIDAD.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

# consistencia con TU espectro (la planilla usa el ancho EFECTIVO, §3.2)
assert abs(CAPACIDAD["r_celda_dl_mbps"] - 2.0 * ESPECTRO["b_dl_ef_mhz"] * 0.78) < 0.5, \
    "r_celda_dl no es SE x B_efectivo x (1-OH) con TUS números — ¿usaste el ancho total? (§3.2)"
assert abs(CAPACIDAD["r_sitio_dl_mbps"] - 3 * CAPACIDAD["r_celda_dl_mbps"]) < 0.5, \
    "un sitio son 3 sectores (§3.2)"

# la demanda sale de TU R5 y TU área
assert abs(CAPACIDAD["demanda_total_mbps"] - REQ["capacidad_mbps_km2"] * REQ["area_km2"]) < 1.0, \
    "demanda_total no es R5 x área del proyecto (§3.3)"

# el veredicto es de los dos dimensionamientos, no de uno
assert CAPACIDAD["n_capacidad"] == _m.ceil(CAPACIDAD["demanda_total_mbps"] / CAPACIDAD["r_sitio_dl_mbps"]), \
    "n_capacidad no sale de tu propia demanda y tu propio sitio (§3.3)"
assert CAPACIDAD["n_sitios"] == max(COBERTURA["n_cobertura"], CAPACIDAD["n_capacidad"]), \
    "el veredicto es max(cobertura, capacidad) (§3.3)"
assert CAPACIDAD["n_sitios"] <= REQ["max_sitios"], \
    "el veredicto excede los sitios que autoriza el municipio"

# la subida no debe mandar (§3.4) — si manda, el veredicto de arriba es inválido
assert abs(CAPACIDAD["ul_instalado_mbps"] - CAPACIDAD["n_sitios"] * CAPACIDAD["r_sitio_ul_mbps"]) < 1.0, \
    "ul_instalado no es n_sitios x sitio UL (§3.4)"
assert CAPACIDAD["ul_instalado_mbps"] >= CAPACIDAD["ul_demanda_mbps"], \
    "la subida instalada no cubre su demanda — el UL mandaría y el veredicto se rehace (§3.4)"

print(f"CAPACIDAD bien formada ✓ — veredicto: {CAPACIDAD['n_sitios']} sitios; "
      f"la Fase 4 los pone en el mapa")


**JUSTIFICACIÓN (Fase 3) — responde aquí mismo:**

1. Compara tu $N_{cobertura}$ con tu $N_{capacidad}$. ¿Qué significa el
   resultado para este proyecto, y en cuánto tiempo se consume tu margen
   si el tráfico crece ~25% al año? (pista: margen = 1.25^t)

   _tu respuesta..._

2. ¿Por qué $SE_{UL} = 1.2 < SE_{DL} = 2.0$? ¿Y por qué ninguno de los
   dos es el 1.0 que usaste en la Fase 2?

   _tu respuesta..._

3. Si la participación de mercado sube diez puntos porcentuales respecto
   de tu escenario de la Fase 0, ¿qué pasa con el veredicto? Muestra la aritmética.

   _tu respuesta..._

4. Tu red instala mucha más capacidad de bajada que de subida. ¿Es un
   defecto del diseño? ¿Cuándo lo sería?

   _tu respuesta..._


---

## Fase 4 — Plan nominal

**Enunciado.** El veredicto de la Fase 3 dijo *cuántos* sitios; aquí
decides **dónde** — sobre TU mapa — y corres el trazador para ver las
consecuencias. Configuración dada: sectorización 3×120° (§4.1), azimuts
0°/120°/240° (§4.2), tilt inicial por sitio, escalado a la altura real de su azotea (§4.3), mapa en calidad de
exploración (10⁵ muestras). **Necesitas el runtime con GPU (T4).**

**Pistas.**

- Posiciones: TU decisión — no hay una respuesta única, se evalúa el
  **criterio** (justificación 1). La celda siguiente imprime los límites
  exactos de TU mapa; tu radio de diseño es `COBERTURA["radio_m"]`.
  Evita lo que §4.2 prohíbe: dos sitios de frente y sitios pegados al borde.
- **Métrica para iterar tu colocación**: los dos proxies que imprime la
  celda (R2 y R3, en calidad exploración). Mueve sitios, corre, compara —
  no es inspección visual: es Sionna como evaluador. Ojo: compara solo
  corridas de la **misma calidad** (§6.1) — el nivel absoluto a 10⁵ está
  dominado por el muestreo (píxeles vacíos cuentan en contra) y subirá
  con el 10⁶ de la Fase 6.
- El % de SINR va a salir LEJOS de R3 — antes de asustarte, lee §4.4
  (qué compró la sectorización) y la nota del elemento de antena en el
  código.


In [ ]:
# ============ TU TRABAJO (Fase 4) — completa las posiciones ============
# OJO: completa TODOS los None de esta celda ANTES de ejecutarla —
# el mapa tarda ~1-3 min por corrida y no querrás correrlo dos veces.
import numpy as np
import matplotlib.pyplot as plt
from sionna.rt import load_scene, Transmitter, PlanarArray, RadioMapSolver

scene = load_scene(REQ["escena"])
scene.frequency = REQ["fc_hz"]
bb = scene.mi_scene.bbox()
print(f"Límites de {ESCENA}: x {float(bb.min[0]):.0f}…{float(bb.max[0]):.0f} m, "
      f"y {float(bb.min[1]):.0f}…{float(bb.max[1]):.0f} m | "
      f"radio de diseño {COBERTURA['radio_m']:.0f} m")

# (dado) Elemento sectorial 3GPP (65°, ~8 dBi). Tu link budget usó 16 dBi
# (elemento + array): el mapa queda ~8 dB conservador — recuérdalo al leerlo.
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5, horizontal_spacing=0.5,
                             pattern="tr38901", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5, horizontal_spacing=0.5,
                             pattern="iso", polarization="V")

# (dado) altura_en(x, y): ray cast vertical — devuelve la altura real del
# techo (o 0.0 si caes en la calle). La antena queda a techo + 3 m de mástil.
import mitsuba as mi
def altura_en(x, y):
    ray = mi.Ray3f(mi.Point3f(float(x), float(y), 500.0), mi.Vector3f(0.0, 0.0, -1.0))
    si = scene.mi_scene.ray_intersect(ray)
    return float(np.array(si.p.z)[0]) if bool(np.array(si.is_valid())[0]) else 0.0


# TU DECISIÓN: elige el EDIFICIO de cada sitio — da solo (x, y); la altura
# sale del techo real via altura_en. Si caes en la calle, la antena queda a
# 3 m: microcelda accidental (el verificador te lo dirá).
# Usa los límites impresos arriba y el plot de huellas para ubicar
# manzanas con edificios; tu radio de diseño: COBERTURA["radio_m"].
SITIOS_XY = {"s1": [None, None],   # <- COMPLETA: x, y
             "s2": [None, None],   # <- COMPLETA: x, y
             "s3": [None, None],   # <- COMPLETA: x, y
             "s4": [None, None]}  # <- COMPLETA: x, y
# Si tu mapa muestra huecos reales, puedes AGREGAR sitios (hasta
# REQ["max_sitios"]): el veredicto de la Fase 3 es el mínimo analítico,
# el trazador manda sobre la geometría. Justifícalo en la pregunta 1.

assert not any(None in xy for xy in SITIOS_XY.values()), \
    "completa las coordenadas x, y de cada sitio"
SITIOS = {n: [float(x), float(y), altura_en(x, y) + 3.0]
          for n, (x, y) in SITIOS_XY.items()}
for n, p in SITIOS.items():
    print(f"{n}: ({p[0]:.0f}, {p[1]:.0f}) — antena a {p[2]:.1f} m (techo + 3)")
AZIMUTS_DEG = [0.0, 120.0, 240.0]   # (dado) §4.2
# (dado) Tilt inicial POR SITIO, escalado a su altura real (§4.3):
# el haz debe tocar suelo a ~0.7 x radio de diseño (mismo criterio con el
# que la lección eligió 6° para 28.5 m) — antena más baja, menos tilt.
d_obj = 0.7 * COBERTURA["radio_m"]
TILT_DEG = {n: float(np.degrees(np.arctan((p[2] - 1.5) / d_obj)))
            for n, p in SITIOS.items()}
for n, t in TILT_DEG.items():
    print(f"tilt {n}: {t:.1f}° (antena a {SITIOS[n][2]:.1f} m)")

for sname, pos in SITIOS.items():
    for k, az in enumerate(AZIMUTS_DEG):
        scene.add(Transmitter(f"{sname}c{k}", position=[float(v) for v in pos],
                              orientation=[float(np.deg2rad(az)),
                                           float(np.deg2rad(TILT_DEG[sname])), 0.0],
                              power_dbm=REQ["p_tx_dbm_max"]))

# (dado) Mapa en calidad de exploración — tarda ~1-3 min en T4
rm = RadioMapSolver()(scene, max_depth=5, cell_size=(5.0, 5.0),
                      samples_per_tx=10**5, diffuse_reflection=True)
sinr_lin  = np.array(rm.sinr).max(axis=0)     # mejor servidor por píxel
pct_sinr0 = None   # <- COMPLETA: % de píxeles con SINR > 0 dB: (sinr_lin > umbral).mean()*100
                   # ojo: sinr_lin está en escala LINEAL — 0 dB equivale a 1.0 (§0.2)


# Métrica gemela: proxy de R2 (control) con la conversión de §2.5 —
# RSRP = RSS + 8 dB (array no modelado, §4.4) - 10log10(N_RE).
# Los píxeles sin cobertura cuentan como incumplimiento.
rss_w    = np.array(rm.rss).max(axis=0)
rss_dbm  = 10*np.log10(np.where(rss_w > 0, rss_w, np.nan)) + 30
rsrp_dbm = rss_dbm + 8.0 - 10*np.log10(ESPECTRO["n_prb"] * 12)
pct_rsrp = float((np.nan_to_num(rsrp_dbm, nan=-999) >= REQ["rsrp_min_dbm"]).mean() * 100)

PLAN = {"sitios": SITIOS, "azimuts_deg": AZIMUTS_DEG, "tilt_deg": TILT_DEG,
        "n_celdas": 3 * len(SITIOS), "pct_sinr0": pct_sinr0,
        "pct_rsrp": pct_rsrp}
# (guardia: solo detecta None pendientes — estos None no se completan)
if pct_sinr0 is None:
    print("(falta pct_sinr0 — calcula el porcentaje y vuelve a ejecutar)")
else:
    print(f"{len(SITIOS)} sitios x 3 sectores = {PLAN['n_celdas']} celdas")
    print(f"proxy R2: RSRP>={REQ['rsrp_min_dbm']:.0f} en {pct_rsrp:.1f}% (meta {REQ['rsrp_prob']:.0%}) | "
          f"proxy R3: SINR>0 en {pct_sinr0:.1f}% (meta {REQ['sinr_prob']:.0%})")


# Huellas de edificios (misma técnica del notebook de la lección)
import mitsuba as mi
from matplotlib.collections import PolyCollection
tris = []
for shape in scene.mi_scene.shapes():
    params = mi.traverse(shape)
    if "vertex_positions" not in params: continue
    V = np.array(params["vertex_positions"]).reshape(-1, 3)
    F = np.array(params["faces"]).reshape(-1, 3)
    if V[:, 2].max() < 1.0: continue          # el suelo no es edificio
    tris.append(V[F][:, :, :2])
tris = np.concatenate(tris)
# (dado) El mapa, para mirar TU diseño
cc = np.array(rm.cell_centers)
extent = [cc[...,0].min(), cc[...,0].max(), cc[...,1].min(), cc[...,1].max()]
sinr_db = 10*np.log10(np.where(sinr_lin > 0, sinr_lin, np.nan))
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sinr_db, origin="lower", extent=extent, cmap="RdYlGn",
               vmin=-10, vmax=30, zorder=2, alpha=0.85)
plt.colorbar(im, shrink=0.8, label="SINR [dB]")
ax.add_collection(PolyCollection(tris, facecolor="0.3", edgecolor="none", zorder=1))
for sname, pos in SITIOS.items():
    ax.plot(pos[0], pos[1], "r^", markersize=12, markeredgecolor="white")
    ax.annotate(sname, (pos[0], pos[1]), textcoords="offset points",
                 xytext=(8, 8), color="white", fontsize=11)
ax.set_title("Tu plan nominal (exploración 10^5)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

In [ ]:
# ======== VERIFICADOR (Fase 4) — ejecuta esta celda SIN modificarla ========
CLAVES_P = {"sitios", "azimuts_deg", "tilt_deg", "n_celdas", "pct_sinr0", "pct_rsrp"}
faltan = CLAVES_P - set(PLAN)
assert not faltan, f"faltan claves en PLAN: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES_P if PLAN.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

# al menos los del veredicto (mínimo analítico), a lo más los del municipio;
# si agregas sitios sobre el veredicto, el mapa debe justificarlo (huecos)
assert CAPACIDAD["n_sitios"] <= len(PLAN["sitios"]) <= REQ["max_sitios"], \
    f"sitios fuera de rango: el veredicto pide {CAPACIDAD['n_sitios']} como " \
    f"mínimo y el municipio autoriza {REQ['max_sitios']}"
assert PLAN["n_celdas"] == 3 * len(PLAN["sitios"]), "3 sectores por sitio (§4.1)"
assert len(scene.transmitters) == PLAN["n_celdas"], \
    "la escena no tiene una celda por sector — revisa el bucle de Transmitters"

# los sitios deben caer DENTRO del mapa, a altura de azotea
bb = scene.mi_scene.bbox()
for nombre, pos in PLAN["sitios"].items():
    x, y, z = pos
    assert None not in pos, f"{nombre}: completa la posición"
    assert bb.min[0] <= x <= bb.max[0] and bb.min[1] <= y <= bb.max[1], \
        f"{nombre} cae fuera del mapa (x: {bb.min[0]:.0f}..{bb.max[0]:.0f}, " \
        f"y: {bb.min[1]:.0f}..{bb.max[1]:.0f})"
    assert z >= 13.0, f"{nombre}: la antena quedó a {z:.0f} m — caíste en la calle o en " \
        "un edificio bajo; búscale una azotea de verdad (plot de huellas)"
    assert z <= 80.0, f"{nombre}: altura sospechosa"

assert all(0.0 <= t <= 15.0 for t in PLAN["tilt_deg"].values()), \
    "tilt inicial fuera de rango (§4.3)"
assert 20.0 < PLAN["pct_sinr0"] <= 100.0, \
    "cobertura SINR>0 sospechosamente baja — ¿sitios enterrados o superpuestos?"
assert 20.0 < PLAN["pct_rsrp"] <= 100.0, "proxy R2 sospechosamente bajo"

print(f"PLAN bien formado ✓ — {len(PLAN['sitios'])} sitios, {PLAN['n_celdas']} celdas | "
      f"proxies: R2 {PLAN['pct_rsrp']:.1f}% / R3 {PLAN['pct_sinr0']:.1f}% "
      f"(la Fase 6 emite el veredicto formal)")


**JUSTIFICACIÓN (Fase 4) — responde aquí mismo:**

1. ¿Con qué criterio colocaste tus sitios, y qué buscaste evitar (§4.2)?

   _tu respuesta..._

2. Tu % de SINR > 0 salió lejos del 90% de R3. ¿Significa que tu diseño
   está mal? Da al menos dos razones (§4.4 y la nota del elemento de
   antena en el código).

   _tu respuesta..._

3. Si sectorizar triplica la capacidad, ¿por qué no usar 6 sectores de
   60° y sextuplicarla (§4.1)?

   _tu respuesta..._

4. El mapa usa un elemento de ~8 dBi y tu link budget usó 16 dBi. ¿Cómo
   debe leerse el mapa al compararlo con R2/R3?

   _tu respuesta..._


---

## Fase 5 — Planificación detallada

**Enunciado.** Tu red ya tiene geometría (Fase 4); ahora los números que
los procedimientos del UE consumen (§5.1): el **PCI** de cada celda por
coloreo mod-3 sobre el grafo de vecindad que sale de TU mapa best-server
(§5.2), la **zona de contención del PRACH** — N_CS desde tu radio de
diseño (§5.3) — y las **tracking areas** (§5.4). Fase analítica: no corre
el trazador, reusa el mapa `rm` de tu Fase 4 (misma sesión).

**Pistas.**

- El grafo de vecindad y el coloreo greedy vienen armados; tú asignas los
  PCI (regla PCI = 3·N₁ + N₂, §5.2) y completas la cadena RACH.
- La tabla de N_CS es la del estándar (TS 38.211); tu radio es
  `COBERTURA["radio_m"]`.


In [ ]:
# ============ TU TRABAJO (Fase 5) — completa cada None ============
# Fase analítica: NO corre el trazador — reusa el mapa rm de tu Fase 4
# (ejecuta esta celda en la MISMA sesión, después de la Fase 4).
# Teoría: lección §5.1-§5.4 y nota crs-dos-celdas-pci-mod3.md.
from collections import Counter

# (dado) mejor servidor por píxel, a partir del mapa rm de la Fase 4
celdas = list(scene.transmitters.keys())            # s1c0 ... sNc2
rss_tx = np.array(rm.rss)                           # (n_celdas, ny, nx)
best = np.where(rss_tx.max(axis=0) > 0, rss_tx.argmax(axis=0), -1)

# (dado) grafo de vecindad: dos celdas son vecinas si sus áreas best-server
# se tocan; el peso es la frontera común en píxeles (píxel = 5 m)
frontera = Counter()
for a, b in ((best[:, :-1], best[:, 1:]), (best[:-1, :], best[1:, :])):
    m = (a != b) & (a >= 0) & (b >= 0)
    for i, j in zip(a[m].ravel(), b[m].ravel()):
        frontera[frozenset((int(i), int(j)))] += 1
vecinos = {i: set() for i in range(len(celdas))}
for par in frontera:
    i, j = tuple(par)
    vecinos[i].add(j); vecinos[j].add(i)

def conflicto_m(grupo):
    """metros de frontera entre celdas vecinas que comparten grupo mod 3"""
    return 5.0 * sum(n for par, n in frontera.items()
                     if len(par) == 2 and len({grupo[i] for i in par}) == 1)

# (dado) coloreo greedy ponderado (§5.2). Restricción DURA: los 3 sectores
# de un sitio usan los 3 grupos mod-3 (siempre son vecinos, en los valles
# de handover). Entre sitios la restricción es blanda: se minimizan los
# metros de frontera en conflicto.
sitio_de = {i: c.rsplit("c", 1)[0] for i, c in enumerate(celdas)}
co_sitio = {i: {j for j in range(len(celdas))
                if j != i and sitio_de[j] == sitio_de[i]}
            for i in range(len(celdas))}
orden = sorted(vecinos, key=lambda i: -sum(
    frontera[frozenset((i, j))] for j in vecinos[i]))
grupo = {}
for i in orden:
    prohibidos = {grupo[j] for j in co_sitio[i] if j in grupo}
    candidatos = [g for g in range(3) if g not in prohibidos]
    costo = {g: sum(frontera[frozenset((i, j))] for j in vecinos[i]
                    if grupo.get(j) == g) for g in candidatos}
    grupo[i] = min(costo, key=costo.get)

# TU DECISIÓN (§5.2): un PCI único por celda, con el mod 3 planificado.
# PCI = 3·N1 + N2 y el mod 3 lo fija la PSS: usa el índice i de la celda
# como N1 y su grupo mod-3 (grupo[i], del coloreo de arriba) como N2.
PCI = {celdas[i]: None for i in range(len(celdas))}   # <- COMPLETA: fórmula con i y grupo[i]
assert not any(v is None for v in PCI.values()), "completa el PCI de cada celda"
print(f"{'celda':<6} {'PCI':>4} {'mod 3':>6}")
for i, c in enumerate(celdas):
    print(f"{c:<6} {PCI[c]:>4} {PCI[c] % 3:>6}")
naive = {i: i % 3 for i in range(len(celdas))}
grupo_final = {i: PCI[celdas[i]] % 3 for i in range(len(celdas))}
print(f"\nfrontera en conflicto mod-3: ingenuo {conflicto_m(naive):,.0f} m "
      f"-> tu plan {conflicto_m(grupo_final):,.0f} m "
      f"(total de fronteras: {5.0*sum(frontera.values()):,.0f} m)")

# ---- RACH (§5.3): la zona de contención debe cubrir el radio de diseño ----
C_LUZ = 3e8
DS_S  = 100e-9   # hipótesis declarada: delay spread urbano (~100 ns; el
                 # notebook de la lección midió 60 ns en San Isidro)
L_ZC, T_SEQ = 839, 800e-6      # preámbulo largo formato 0 (TS 38.211)
NCS_TABLA = [13, 15, 18, 22, 26, 32, 38, 46, 59, 76, 93, 119, 167, 279, 419]

t_guard = None   # <- COMPLETA: ida y vuelta del UE del borde + delay spread:
                 # 2·radio/C_LUZ + DS_S — tu radio: COBERTURA["radio_m"]
ncs_min = None   # <- COMPLETA: en chips de la secuencia: int(np.ceil(t_guard/T_SEQ * L_ZC))
assert None not in (t_guard, ncs_min), "completa t_guard y ncs_min"  # guardia: no tocar
ncs = next(n for n in NCS_TABLA if n >= ncs_min)  # (dado) menor N_CS que alcanza
preambulos_por_raiz = None   # <- COMPLETA: desplazamientos que caben en la raíz: L_ZC // ncs
raices = None   # <- COMPLETA: cada celda anuncia 64 preámbulos:
                             # int(np.ceil(64/preambulos_por_raiz))
print(f"RACH: radio {COBERTURA['radio_m']:.0f} m -> t_guard {t_guard*1e6:.2f} µs "
      f"-> N_CS mínimo {ncs_min} -> N_CS={ncs} -> {preambulos_por_raiz} "
      f"preámbulos/raíz -> {raices} raíz(es) por celda")

# ---- Tracking areas (§5.4): TU DECISIÓN ----
n_tracking_areas = None   # <- COMPLETA: ¿cuántas TAs para ~1.3 km² con una zona de paging?
assert None not in (preambulos_por_raiz, raices, n_tracking_areas), \
    "completa preambulos_por_raiz, raices y n_tracking_areas"

DETALLE = {"pci": PCI, "grupo_mod3": {c: PCI[c] % 3 for c in PCI},
           "conflicto_m": conflicto_m(grupo_final),
           "rach_ncs": ncs, "rach_preambulos_por_raiz": preambulos_por_raiz,
           "rach_raices_por_celda": raices,
           "n_tracking_areas": n_tracking_areas}
print("\nDETALLE listo — la Fase 6 valida el plan completo contra el mapa consolidado.")

In [ ]:
# ======== VERIFICADOR (Fase 5) — ejecuta esta celda SIN modificarla ========
CLAVES_D = {"pci", "grupo_mod3", "conflicto_m", "rach_ncs",
            "rach_preambulos_por_raiz", "rach_raices_por_celda",
            "n_tracking_areas"}
faltan = CLAVES_D - set(DETALLE)
assert not faltan, f"faltan claves en DETALLE: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES_D if DETALLE.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

pci_v = DETALLE["pci"]
assert set(pci_v) == set(scene.transmitters.keys()), \
    "falta (o sobra) el PCI de alguna celda de tu escena"
assert all(0 <= int(v) <= 1007 for v in pci_v.values()), \
    "PCI fuera del rango 0..1007 (§5.2)"
assert len(set(pci_v.values())) == len(pci_v), \
    "PCIs repetidos — colisión asegurada (§5.2, regla 1)"
assert set(DETALLE["grupo_mod3"]) == set(pci_v), \
    "grupo_mod3 debe tener exactamente las mismas celdas que pci"
assert all(DETALLE["grupo_mod3"][c] == pci_v[c] % 3 for c in pci_v), \
    "grupo_mod3 no es PCI mod 3 — el grupo lo fija la PSS (§5.2)"

# regla DURA: los 3 sectores de cada sitio en grupos mod-3 distintos
for s in {c.rsplit("c", 1)[0] for c in pci_v}:
    gr = {pci_v[c] % 3 for c in pci_v if c.rsplit("c", 1)[0] == s}
    assert len(gr) == 3, \
        f"los 3 sectores de {s} comparten grupo mod 3 — son vecinos " \
        "garantizados en los valles de handover (§5.2)"

esperado_m = conflicto_m({i: pci_v[celdas[i]] % 3 for i in range(len(celdas))})
assert DETALLE["conflicto_m"] == esperado_m, \
    "conflicto_m no corresponde a TU plan de PCI — debe salir de " \
    "conflicto_m(...) sobre los grupos mod-3 de tu plan"

# RACH: N_CS estándar y suficiente para el radio de diseño
assert DETALLE["rach_ncs"] in NCS_TABLA, \
    "N_CS debe ser un valor de la tabla del estándar (TS 38.211)"
ncs_suf = (2 * COBERTURA["radio_m"] / 3e8 + DS_S) / T_SEQ * L_ZC
assert DETALLE["rach_ncs"] >= ncs_suf, \
    f"N_CS={DETALLE['rach_ncs']} no cubre el radio de diseño — un UE " \
    "legítimo del borde caería como OTRO preámbulo (§5.3)"
assert DETALLE["rach_preambulos_por_raiz"] == L_ZC // DETALLE["rach_ncs"], \
    "preámbulos por raíz inconsistente con tu N_CS (§5.3)"
import math
assert DETALLE["rach_raices_por_celda"] == \
    math.ceil(64 / DETALLE["rach_preambulos_por_raiz"]), \
    "raíces por celda inconsistente: cada celda anuncia 64 preámbulos (§5.3)"

assert DETALLE["n_tracking_areas"] == 1, \
    "esta red cabe entera en UNA tracking area: ~1.3 km² y una sola " \
    "zona de paging — partirla solo agrega TAUs (§5.4)"

print(f"DETALLE bien formado ✓ — {len(pci_v)} PCIs únicos, regla co-sitio OK | "
      f"conflicto mod-3: {DETALLE['conflicto_m']:,.0f} m | "
      f"N_CS={DETALLE['rach_ncs']} -> {DETALLE['rach_raices_por_celda']} raíz/celda | "
      f"{DETALLE['n_tracking_areas']} TA "
      f"(la Fase 6 emite el veredicto formal)")

**JUSTIFICACIÓN (Fase 5) — responde aquí mismo:**

1. ¿Por qué la regla co-sitio (3 sectores en los 3 grupos mod-3) es dura,
   y la regla entre sitios es blanda (§5.2)?

   _tu respuesta..._

2. Con tu radio de diseño el N_CS salió el mínimo de la tabla. ¿Qué cambia
   en una celda rural de 15 km, y por qué el plan de raíces se hace junto
   al de PCI (§5.3)?

   _tu respuesta..._

3. ¿Por qué una sola tracking area aquí (si eso decidiste), y qué costo
   paga quien exagera en cada dirección (§5.4)?

   _tu respuesta..._

4. El verificador acepta tu plan aunque `conflicto_m` > 0. ¿Por qué un
   plan con fronteras mod-3 en conflicto sigue siendo válido (§5.2)?

   _tu respuesta..._

---

## Fase 6 — Validación: el veredicto R1–R8

**Enunciado.** Cada fase dejó hipótesis declaradas (SE 2.0, σ = 8, el
veredicto analítico de sitios); aquí se cobran contra el **mapa
consolidado** (10⁶ rayos + difusa, §6.1) y el diseño se valida requisito
por requisito. Tres celdas: el mapa (cara — córrela UNA sola vez), tus
verificaciones (analíticas, re-ejecutables gratis) y el bisturí de tilt
más el veredicto final.

**Pistas.**

- R2 y R3 se miden sobre el área TOTAL (los píxeles sin cobertura cuentan
  en contra); R4-DL sobre los píxeles cubiertos (§6.3).
- El mapa SINR está en escala LINEAL; los umbrales del TdR, en dB (§0.2).
- Que repruebe alguna meta de radio no es "tu nota": es el estado normal
  de un plan nominal al salir de la validación — la justificación 4 pide
  tus salidas en orden de costo (§6.5).


In [ ]:
# ========= MAPA CONSOLIDADO (Fase 6) — ejecuta UNA sola vez (~10-30 min GPU) =========
# El mapa "de reporte": 10^6 rayos + reflexión difusa, 10x tu exploratorio.
# Todas las verificaciones de esta fase salen de aquí. Regla permanente:
# los mapas solo se comparan ENTRE LA MISMA calidad de muestreo (§6.1).
rm6 = RadioMapSolver()(scene, max_depth=5, cell_size=(5.0, 5.0),
                       samples_per_tx=10**6, diffuse_reflection=True)
rss6  = np.array(rm6.rss)                       # [n_celdas, ny, nx] W
sinr6 = np.array(rm6.sinr)
cubierto6 = rss6.sum(axis=0) > 0
best6 = np.where(cubierto6, rss6.argmax(axis=0), -1)
pct_sinr0_cons = float((sinr6.max(axis=0) > 1.0).mean() * 100)
print(f"consolidado 10^6: SINR>0 dB en {pct_sinr0_cons:.1f}% del área "
      f"(tu exploratorio 10^5 daba {PLAN['pct_sinr0']:.1f}% — la diferencia "
      f"es muestreo, no red: compara cada mapa solo con su propia calidad)")

In [ ]:
# ============ TU TRABAJO (Fase 6) — completa cada None ============
# Analítico sobre rm6: re-ejecutable sin costo (el mapa ya está en memoria).
# --- R2 (§2.5, §6.2): RSS -> RSRP con la corrección de array declarada ---
G_ARRAY_DB = 8.0   # (dado) el solver modela el elemento (~8 dBi); tu link
                   # budget usó 16 (elemento + array 4x1): corrección declarada
N_RE = ESPECTRO["n_prb"] * 12
rss6_srv_dbm = 10*np.log10(np.where(cubierto6, rss6.max(axis=0), np.nan)) + 30
rsrp6_dbm = None   # <- COMPLETA: RSRP = RSS del servidor + G_ARRAY_DB - 10log10(N_RE)  (§2.5)
pct_r2 = None   # <- COMPLETA: % del área TOTAL con RSRP >= REQ["rsrp_min_dbm"] — los píxeles
                   # sin cobertura (NaN) cuentan como incumplimiento:
                   # float((np.nan_to_num(rsrp6_dbm, nan=-999) >= umbral).mean()*100)
assert rsrp6_dbm is not None and pct_r2 is not None, "completa rsrp6_dbm y pct_r2"

# --- R3: calidad de datos sobre el área total ---
sinr6_srv = np.where(cubierto6, sinr6.max(axis=0), 0.0)     # (dado) LINEAL
pct_r3 = None   # <- COMPLETA: % del área con SINR sobre REQ["sinr_min_db"] — ojo: el mapa
                   # es LINEAL y el umbral está en dB (§0.2): umbral_lin = 10**(dB/10)
assert pct_r3 is not None, "completa pct_r3"  # guardia: no tocar

# --- SE medida (§6.3): media espacial por celda, techo 256-QAM ---
SE_MAX = 7.4       # (dado) techo de 256-QAM en bit/s/Hz
se_pix = None   # <- COMPLETA: por píxel: min(log2(1 + SINR_lineal), SE_MAX)
                   # -> np.minimum(np.log2(1 + sinr6_srv), SE_MAX)
assert se_pix is not None, "completa se_pix"  # guardia: no tocar
se_por_celda = [float(se_pix[best6 == i].mean())
                for i in range(len(celdas)) if (best6 == i).any()]  # (dado)
SE_MEDIDA = None   # <- COMPLETA: media de la red: float(np.mean(se_por_celda))

# --- Re-veredicto de capacidad con la SE medida (fórmulas §3.2-3.3) ---
r_celda_med_mbps = None   # <- COMPLETA: SE_MEDIDA * ESPECTRO["b_dl_ef_mhz"] * (1 - oh)
n_cap_med = None   # <- COMPLETA: math.ceil(demanda_total / capacidad de sitio) — usa
                          # CAPACIDAD["demanda_total_mbps"] y 3 celdas por sitio

# --- R4-DL: throughput de borde sobre los píxeles CUBIERTOS ---
thr_pix_mbps = se_pix * ESPECTRO["b_dl_ef_mhz"] * (1 - oh)   # (dado)
thr_p5 = None   # <- COMPLETA: percentil 5: float(np.percentile(thr_pix_mbps[cubierto6], 5))
assert None not in (SE_MEDIDA, r_celda_med_mbps, n_cap_med, thr_p5), \
    "completa SE_MEDIDA, r_celda_med_mbps, n_cap_med y thr_p5"

print(f"R2: RSRP >= {REQ['rsrp_min_dbm']:.0f} dBm en {pct_r2:.1f}% del área total "
      f"(meta {REQ['rsrp_prob']:.0%}) | RSRP p50 {np.nanpercentile(rsrp6_dbm, 50):.1f} "
      f"/ p5 {np.nanpercentile(rsrp6_dbm, 5):.1f} dBm")
print(f"R3: SINR >= {REQ['sinr_min_db']:.0f} dB en {pct_r3:.1f}% del área "
      f"(meta {REQ['sinr_prob']:.0%})")
print(f"SE medida = {SE_MEDIDA:.2f} bit/s/Hz (hipótesis F3: 2.0) -> celda "
      f"{r_celda_med_mbps:.0f} Mbps -> {n_cap_med} sitio(s) por capacidad "
      f"(la planilla F3 daba {CAPACIDAD['n_capacidad']})")
print(f"R4-DL: throughput p5 = {thr_p5:.0f} Mbps (meta {REQ['thr_borde_dl_mbps']:.0f}) "
      f"| R4-UL por cálculo: MAPL_UL {COBERTURA['mapl_ul_db']:.0f} > "
      f"MAPL_DL {COBERTURA['mapl_dl_db']:.0f} dB -> el UL no limita")

# --- Drive test virtual (§6.2), dado: n y sigma medidos del propio mapa ---
# PL aparente = P_tx + ganancias - RSS por píxel, contra log10(d al sitio
# servidor). Pendiente/10 = n medido; std de residuos = sigma aparente.
# Caveat declarado: el patrón de antena contamina el residuo -> la sigma
# aparente es cota superior del shadowing puro.
cc6 = np.array(rm6.cell_centers)                    # [ny, nx, 3]
nombres_sitios = list(PLAN["sitios"])
pos_sitios = np.array([PLAN["sitios"][s] for s in nombres_sitios])
sitio_idx = np.array([nombres_sitios.index(c.rsplit("c", 1)[0]) for c in celdas])
d_srv = np.linalg.norm(
    cc6 - pos_sitios[sitio_idx[np.clip(best6, 0, len(celdas)-1)]], axis=-1)
m6 = cubierto6 & (d_srv > 30)                       # excluir campo cercano
pl_ap = REQ["p_tx_dbm_max"] + 8.0 + G_ARRAY_DB - rss6_srv_dbm   # +8 dBi elemento
x6, y6 = np.log10(d_srv[m6]), pl_ap[m6]
pend6, ordo6 = np.polyfit(x6, y6, 1)
n_med, sigma_med = float(pend6/10), float(np.std(y6 - (pend6*x6 + ordo6)))
print(f"drive test virtual: n medido = {n_med:.2f} (declarado {N_PROP}) | "
      f"sigma aparente = {sigma_med:.1f} dB (declarada 8)")

In [ ]:
# ========= TILT POR CELDA (Fase 6) — el ajuste fino, medido (§6.4) =========
# Línea base = tu mapa exploratorio de la Fase 4 (rm, 10^5, tilt por sitio).
# El delta se mide a la MISMA calidad (otro mapa 10^5, ~1-3 min) — nunca
# contra rm6 (§6.1). El "invasor" se elige del mapa: la celda cuyos píxeles
# best-server quedan más lejos de su sitio (percentil 90 de distancia).
rss4 = np.array(rm.rss)
cubierto4 = rss4.sum(axis=0) > 0
best4 = np.where(cubierto4, rss4.argmax(axis=0), -1)
cc4 = np.array(rm.cell_centers)
d4 = np.linalg.norm(
    cc4 - pos_sitios[sitio_idx[np.clip(best4, 0, len(celdas)-1)]], axis=-1)
p90_d = [float(np.percentile(d4[best4 == i], 90)) if (best4 == i).any() else 0.0
         for i in range(len(celdas))]
invasor = int(np.argmax(p90_d))
sitio_inv = celdas[invasor].rsplit("c", 1)[0]
print(f"invasor: {celdas[invasor]} (p90 de distancia best-server = "
      f"{p90_d[invasor]:.0f} m; su sitio: {sitio_inv}, "
      f"tilt del plan {PLAN['tilt_deg'][sitio_inv]:.1f} grados)")

# TU DECISIÓN (§6.4): el tilt del bisturí para el invasor — más agresivo
# que el del plan (p.ej. el tilt de su sitio + 6 grados)
tilt_bist_deg = None   # <- COMPLETA: p.ej. PLAN["tilt_deg"][sitio_inv] + 6
assert tilt_bist_deg is not None, "completa tilt_bist_deg"  # guardia: no tocar
k_inv = int(celdas[invasor][-1])
scene.get(celdas[invasor]).orientation = [float(np.deg2rad(AZIMUTS_DEG[k_inv])),
                                          float(np.deg2rad(tilt_bist_deg)), 0.0]
rm_b = RadioMapSolver()(scene, max_depth=5, cell_size=(5.0, 5.0),
                        samples_per_tx=10**5, diffuse_reflection=True)
pct_bist = float((np.array(rm_b.sinr).max(axis=0) > 1.0).mean() * 100)
delta_pp = pct_bist - PLAN["pct_sinr0"]
# métrica local: SINR medio donde el invasor sirve lejos (> 300 m)
lejos = (best4 == invasor) & (d4 > 300)
s_base = 10*np.log10(np.maximum(np.array(rm.sinr).max(axis=0), 1e-9))
s_bist = 10*np.log10(np.maximum(np.array(rm_b.sinr).max(axis=0), 1e-9))
print(f"global: SINR>0 {PLAN['pct_sinr0']:.1f}% -> {pct_bist:.1f}% "
      f"(delta {delta_pp:+.1f} pp)")
if lejos.any():
    print(f"local : SINR medio en la zona lejana del invasor (>300 m): "
          f"{s_base[lejos].mean():.1f} -> {s_bist[lejos].mean():.1f} dB")
# restaurar el plan nominal
scene.get(celdas[invasor]).orientation = [
    float(np.deg2rad(AZIMUTS_DEG[k_inv])),
    float(np.deg2rad(PLAN["tilt_deg"][sitio_inv])), 0.0]

# ---- Veredicto final R1-R8 (dado): tus números contra el contrato ----
filas = [
    ("R1", "área de servicio", "la escena cubre el área del TdR", True),
    ("R2", f"RSRP>={REQ['rsrp_min_dbm']:.0f} en {REQ['rsrp_prob']:.0%}",
     f"{pct_r2:.1f}% medido (10^6 + corrección array)",
     bool(pct_r2 >= 100*REQ["rsrp_prob"])),
    ("R3", f"SINR>={REQ['sinr_min_db']:.0f} en {REQ['sinr_prob']:.0%}",
     f"{pct_r3:.1f}% medido", bool(pct_r3 >= 100*REQ["sinr_prob"])),
    ("R4", f"p5 >= {REQ['thr_borde_dl_mbps']:.0f}/{REQ['thr_borde_ul_mbps']:.0f} Mbps",
     f"DL p5={thr_p5:.0f} Mbps; UL por cálculo "
     f"({COBERTURA['mapl_ul_db']:.0f}>{COBERTURA['mapl_dl_db']:.0f} dB)",
     bool(thr_p5 >= REQ["thr_borde_dl_mbps"])),
    ("R5", f">= {REQ['capacidad_mbps_km2']:.0f} Mbps/km2",
     f"SE medida {SE_MEDIDA:.1f} -> {3*r_celda_med_mbps:.0f} Mbps/sitio, "
     f"{n_cap_med} sitio(s) por capacidad",
     bool(n_cap_med <= len(PLAN["sitios"]))),
    ("R6", "eMBB+VoNR, <20 ms", "por arquitectura/QoS (no simulable aquí)", None),   # None = sin veredicto, no tocar
    ("R7", f"{REQ['bw_hz']/1e6:.0f} MHz {REQ['banda']}", "por licencia (dado)", True),
    ("R8", f"max {REQ['max_sitios']} sitios, azoteas",
     f"{len(PLAN['sitios'])} sitios en azoteas (verificado F4)", True),
]
print(f"\n{'req':<4} {'meta':<26} {'evidencia':<58} {'veredicto'}")
for r, meta, ev, ok in filas:
    v = "—" if ok is None else ("CUMPLE" if ok else "NO CUMPLE")   # None -> "—" (dado)
    print(f"{r:<4} {meta:<26} {ev:<58} {v}")
fallan = [r for r, _, _, ok in filas if ok is not None and not ok]   # (dado)
print(f"\nreprueban: {', '.join(fallan) if fallan else 'ninguno'}")

FINAL = {"pct_sinr0_cons": pct_sinr0_cons, "pct_r2": pct_r2, "pct_r3": pct_r3,
         "thr_p5_mbps": thr_p5, "se_medida": SE_MEDIDA, "n_cap_med": n_cap_med,
         "n_medido": n_med, "sigma_medida": sigma_med,
         "invasor": celdas[invasor], "delta_bisturi_pp": float(delta_pp)}
print("FINAL listo — el diseño queda validado contra R1-R8.")

In [ ]:
# ======== VERIFICADOR (Fase 6) — ejecuta esta celda SIN modificarla ========
import math
CLAVES_F = {"pct_sinr0_cons", "pct_r2", "pct_r3", "thr_p5_mbps", "se_medida",
            "n_cap_med", "n_medido", "sigma_medida", "invasor",
            "delta_bisturi_pp"}
faltan = CLAVES_F - set(FINAL)
assert not faltan, f"faltan claves en FINAL: {sorted(faltan)}"
vacios = sorted(k for k in CLAVES_F if FINAL.get(k) is None)
assert not vacios, f"claves sin valor: {vacios}"

# consistencia contra el mapa consolidado (recomputo exacto, sin regalar):
_rsrp = (10*np.log10(np.where(cubierto6, rss6.max(axis=0), np.nan)) + 30
         + 8.0 - 10*np.log10(ESPECTRO["n_prb"] * 12))
_r2 = float((np.nan_to_num(_rsrp, nan=-999) >= REQ["rsrp_min_dbm"]).mean() * 100)
assert abs(FINAL["pct_r2"] - _r2) < 0.05, \
    "pct_r2 no sale del consolidado con la conversión de §2.5 — revisa la " \
    "corrección de array, el -10log10(N_RE) y que los NaN cuenten como incumplimiento"

_sinr = np.where(cubierto6, sinr6.max(axis=0), 0.0)
_r3 = float((_sinr > 10**(REQ["sinr_min_db"]/10)).mean() * 100)
assert abs(FINAL["pct_r3"] - _r3) < 0.05, \
    "pct_r3 inconsistente con el mapa — ojo: el mapa es LINEAL y el umbral " \
    "está en dB (§0.2)"

_se = np.minimum(np.log2(1 + _sinr), 7.4)
_se_med = float(np.mean([_se[best6 == i].mean()
                         for i in range(len(celdas)) if (best6 == i).any()]))
assert abs(FINAL["se_medida"] - _se_med) < 0.02, \
    "se_medida inconsistente — media espacial POR CELDA (best-server), " \
    "techo 7.4 de 256-QAM (§6.3)"

_r_celda = FINAL["se_medida"] * ESPECTRO["b_dl_ef_mhz"] * (1 - oh)
assert FINAL["n_cap_med"] == math.ceil(
    CAPACIDAD["demanda_total_mbps"] / (3 * _r_celda)), \
    "n_cap_med inconsistente: demanda total sobre capacidad de sitio con la " \
    "SE medida, redondeado hacia arriba (§3.3)"

_thr5 = float(np.percentile(_se[cubierto6] * ESPECTRO["b_dl_ef_mhz"] * (1 - oh), 5))
assert abs(FINAL["thr_p5_mbps"] - _thr5) < 0.5, \
    "thr_p5_mbps inconsistente — percentil 5 sobre los píxeles CUBIERTOS (§6.3)"

assert 20.0 < FINAL["pct_sinr0_cons"] <= 100.0, "pct_sinr0_cons sospechoso"
assert 0.5 <= FINAL["n_medido"] <= 6.0, \
    "n medido fuera de rango — ¿dividiste la pendiente entre 10?"
assert 3.0 <= FINAL["sigma_medida"] <= 25.0, "sigma medida fuera de rango"
assert FINAL["invasor"] in set(scene.transmitters.keys()), \
    "invasor debe ser una celda de tu escena"
assert abs(FINAL["delta_bisturi_pp"]) <= 30.0, \
    "delta del bisturí sospechoso — ¿comparaste mapas de la MISMA calidad? (§6.1)"

print(f"FINAL bien formado ✓ — R2 {FINAL['pct_r2']:.1f}% | R3 {FINAL['pct_r3']:.1f}% | "
      f"p5 {FINAL['thr_p5_mbps']:.0f} Mbps | SE {FINAL['se_medida']:.2f} | "
      f"n {FINAL['n_medido']:.2f} | sigma {FINAL['sigma_medida']:.1f} dB | "
      f"bisturí {FINAL['delta_bisturi_pp']:+.1f} pp — examen completo")

**JUSTIFICACIÓN (Fase 6) — responde aquí mismo:**

1. El mismo plan dio dos porcentajes distintos de SINR > 0 (tu exploratorio
   10⁵ y el consolidado 10⁶). ¿Cuál va al veredicto, y por qué no se
   comparan entre sí (§6.1)?

   _tu respuesta..._

2. La SE medida quedó lejos de la hipótesis 2.0 de la Fase 3. ¿Estaba mal
   la hipótesis? ¿Qué ignora cada número, y cuál usarías para dimensionar
   la próxima red (§6.3)?

   _tu respuesta..._

3. ¿Qué delta global midió tu bisturí de tilt, y qué dice el SINR local de
   la zona "invadida"? ¿Concluyes que el tilt es inútil (§6.4)?

   _tu respuesta..._

4. Con tu tabla R1–R8: ¿qué requisitos reprueban (si alguno) y cuáles son
   tus salidas, en orden de costo (§6.5)? Menciona qué dice la σ medida
   sobre el margen de shadowing de la Fase 2 (§6.2).

   _tu respuesta..._

---

*(Fin del examen — entrega el notebook ejecutado de punta a punta, con
todos los verificadores en verde y las justificaciones respondidas.)*